<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260406_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile ex.js

// ================================================================
// 트리를 이용한 데이터베이스 시뮬레이션
// 주제: 토스뱅크 고객 주소록
// ================================================================

// --- Base Structure ---
// 트리 노드 생성 함수
// value: 고객 정보 문자열, left/right: 자식 노드 (초기값 null)
function createNode(value) {
  return {
    value: value,
    left: null,
    right: null
  };
}

// 루트 노드부터 수동으로 트리 구성
// 구조:
//              김민준(서울·예금)
//             /                 \
//     이수빈(부산·적금)       박지훈(대구·대출)
//     /            \
// 최유진(제주·예금)  정하은(광주·적금)
let root = createNode('김민준(서울·예금)');
root.left = createNode('이수빈(부산·적금)');
root.right = createNode('박지훈(대구·대출)');
root.left.left = createNode('최유진(제주·예금)');
root.left.right = createNode('정하은(광주·적금)');

// --- Directory Style Tree Visualization ---
// DFS 재귀로 트리를 디렉토리 형태로 출력
// 오른쪽 자식 -> 현재 노드 -> 왼쪽 자식 순 (90도 회전된 트리)
function printDirectoryTree(node, prefix = "", isLeft = true) {
  if (!node) return;

  // [1] 오른쪽 자식 먼저 출력 (화면 위쪽)
  if (node.right) {
    printDirectoryTree(node.right, prefix + (isLeft ? "│   " : "    "), false);
  }

  // [2] 현재 노드 출력
  console.log(prefix + (isLeft ? "└── " : "┌── ") + node.value);

  // [3] 왼쪽 자식 출력 (화면 아래쪽)
  if (node.left) {
    printDirectoryTree(node.left, prefix + (isLeft ? "    " : "│   "), true);
  }
}

// --- SQL Engine Emulation ---

// SELECT: BFS로 레벨 순 탐색 -> target과 일치하는 노드 반환
function selectNode(target) {
  if (!root) return null;
  const queue = [root];
  while (queue.length > 0) {
    let node = queue.shift();               // 큐 앞에서 꺼냄 (FIFO)
    if (node.value === target) return node; // 일치하면 즉시 반환
    if (node.left) queue.push(node.left);
    if (node.right) queue.push(node.right);
  }
  return null; // 못 찾으면 null
}

// SQL 실행 함수: action에 따라 SELECT/INSERT/UPDATE/DELETE 분기
function executeSQL(action, params) {
  console.log(`\n>> EXECUTING SQL: ${action} ${JSON.stringify(params)}`);

  // SELECT 제외하고 실행 전 트리 상태 출력
  if (action !== 'SELECT') {
    console.log(`[${action} BEFORE]`);
    printDirectoryTree(root);
  }

  switch (action) {

    // SELECT: 고객 탐색
    case 'SELECT':
      console.log(`[SQL Log] SELECT * FROM customer_book WHERE name_info = '${params.target}';`);
      const found = selectNode(params.target);
      console.log(found ? `   - Result: Found ${found.value}` : "   - Result: Not Found");
      break;

    // INSERT: BFS로 빈 자리 찾아 새 고객 노드 삽입 -> 트리 균형 유지
    case 'INSERT':
      console.log(`[SQL Log] INSERT INTO customer_book (name_info) VALUES ('${params.value}');`);
      const newNode = createNode(params.value);
      const queue = [root];
      while (queue.length > 0) {
        let node = queue.shift();
        if (!node.left) { node.left = newNode; break; }   // 왼쪽 빈 자리 -> 삽입
        else queue.push(node.left);
        if (!node.right) { node.right = newNode; break; } // 오른쪽 빈 자리 -> 삽입
        else queue.push(node.right);
      }
      break;

    // UPDATE: 기존 고객 정보 수정 (거주지 변경, 상품 변경 등)
    case 'UPDATE':
      console.log(`[SQL Log] UPDATE customer_book SET name_info = '${params.newValue}' WHERE name_info = '${params.oldValue}';`);
      const targetNode = selectNode(params.oldValue);
      if (targetNode) targetNode.value = params.newValue; // 참조로 직접 수정
      break;

    // DELETE: BFS로 부모 찾은 뒤 해당 자식 연결을 null로 끊음 -> 고객 탈퇴
    case 'DELETE':
      console.log(`[SQL Log] DELETE FROM customer_book WHERE name_info = '${params.target}';`);
      const dQueue = [root];
      while (dQueue.length > 0) {
        let node = dQueue.shift();
        if (node.left && node.left.value === params.target) { node.left = null; break; }
        if (node.right && node.right.value === params.target) { node.right = null; break; }
        if (node.left) dQueue.push(node.left);
        if (node.right) dQueue.push(node.right);
      }
      break;
  }

  // SELECT 제외하고 실행 후 트리 상태 출력
  if (action !== 'SELECT') {
    console.log(`[${action} AFTER]`);
    printDirectoryTree(root);
  }
  console.log("--------------------------------------------------");
}

// --- 시나리오 실행 ---
console.log("===== 토스뱅크 고객 주소록 DB 시뮬레이션 Start =====");

// [1] 최유진(제주·예금) 고객 조회
executeSQL('SELECT', { target: '최유진(제주·예금)' });

// [2] 박지훈 고객 대구 -> 인천으로 거주지 변경
executeSQL('UPDATE', { oldValue: '박지훈(대구·대출)', newValue: '박지훈(인천·대출)' });

// [3] 최유진 고객 탈퇴 처리
executeSQL('DELETE', { target: '최유진(제주·예금)' });

// [4] 신규 고객 홍서연 가입 -> BFS로 빈 자리 찾아 삽입
executeSQL('INSERT', { value: '홍서연(강원·예금)' });

// [5] 존재하지 않는 고객 조회 -> Not Found 확인
executeSQL('SELECT', { target: '김없음(없음·없음)' });

console.log("===== 토스뱅크 고객 주소록 DB 시뮬레이션 End =====");

Writing ex.js


In [2]:
!node ex.js

===== 토스뱅크 고객 주소록 DB 시뮬레이션 Start =====

>> EXECUTING SQL: SELECT {"target":"최유진(제주·예금)"}
[SQL Log] SELECT * FROM customer_book WHERE name_info = '최유진(제주·예금)';
   - Result: Found 최유진(제주·예금)
--------------------------------------------------

>> EXECUTING SQL: UPDATE {"oldValue":"박지훈(대구·대출)","newValue":"박지훈(인천·대출)"}
[UPDATE BEFORE]
│   ┌── 박지훈(대구·대출)
└── 김민준(서울·예금)
    │   ┌── 정하은(광주·적금)
    └── 이수빈(부산·적금)
        └── 최유진(제주·예금)
[SQL Log] UPDATE customer_book SET name_info = '박지훈(인천·대출)' WHERE name_info = '박지훈(대구·대출)';
[UPDATE AFTER]
│   ┌── 박지훈(인천·대출)
└── 김민준(서울·예금)
    │   ┌── 정하은(광주·적금)
    └── 이수빈(부산·적금)
        └── 최유진(제주·예금)
--------------------------------------------------

>> EXECUTING SQL: DELETE {"target":"최유진(제주·예금)"}
[DELETE BEFORE]
│   ┌── 박지훈(인천·대출)
└── 김민준(서울·예금)
    │   ┌── 정하은(광주·적금)
    └── 이수빈(부산·적금)
        └── 최유진(제주·예금)
[SQL Log] DELETE FROM customer_book WHERE name_info = '최유진(제주·예금)';
[DELETE AFTER]
│   ┌── 박지훈(인천·대출)
└── 김민준(서울·예금)
    │   ┌── 정하은(광주·적금)
    └──